# SAC Irrigation Training - v2.17-P3 (exploration injection: alpha=0.005 + decaying action noise)

**Algorithm:** SAC (stable_baselines3) with VDN-factorised twin-Q + LayerNorm critic — architecturally BYTE-IDENTICAL to v2.16.
**The only changes vs v2.16 are training-time:**
1. `ent_coef`: 0.01 -> **0.005** (partial release of the entropy mu-centering "action floor").
2. **Decaying symmetric exploration noise** N(0, sigma) added at collection time, sigma 0.30 -> 0.0 over the first 60k steps.

Marker is unchanged (2.16), so `runner.py` evaluates these checkpoints with ZERO eval-side changes (RAIN_REF=30 applied automatically).

## Why v2.17-P3 exists (the buffer-coverage hypothesis)

The v2.14-v2.16 post-mortem established that the wet-year defect is chronic **over-irrigation**, not rain-blindness:

| quantity (wet/100) | v2.16-fixed | v2.16 auto-alpha | MPC |
|---|---|---|---|
| x1 median (mm) | 152 | 132 | 130 |
| % cell-days > FC | 84% | 36% | 20% |
| waterlog days/agent | 76 | 35 | 18 |
| u_mean (mm) | 4.83 | 3.46 | 3.34 |

Key facts that rule OUT the usual suspects:
- **Reward is not the cause.** Season-sum r6 (waterlog) ~= -13.7 vs r1 (biomass) ~= +1.27 — waterlog already dominates the return ~10x. r6 is linear (matches the ABM's linear h6).
- **Policy is near open-loop in water.** dry/100 u_mean 5.20 vs wet/100 u_mean 4.83 — only 0.37 mm less despite rain. The budget clip, not the policy, sets total water.
- **The actor never voluntarily goes low.** Fraction of cell-days with u < 0.5 mm *while budget remains* = 0.00 across all 9 scenarios.
- The v2.7 change-spec independently recorded near-constant ~5.13 mm/day output (CV 0.055); on a 35.8 mm rain day SAC delivered 5.79 mm while MPC delivered 0.28 mm.

**Hypothesis:** the policy never SAMPLES low-water actions in unconstrained wet states, so the critic never learns they are good, so the policy never moves (buffer-coverage starvation). Injecting decaying exploration noise should populate the buffer with low-water-in-wet transitions and pull wet-year x1 down.

## Acceptance criteria (decide on x1/waterlog, NOT corr or yield)

PRIMARY: wet-year (mean of 3 budgets) x1 median < 140 mm AND waterlog < 55 days (v2.16-fixed: 152 / 76; auto-alpha demo: 132 / 35; MPC: 130 / ~20).
SECONDARY: wet-year water < 360 mm; wet/100 u_mean clearly < dry/100 u_mean.
COVERAGE CHECK (interpretability): `p3/frac_low_action_wet` > 0 during exploration — proves the buffer actually received the transitions.
STABILITY: critic_loss < 100 throughout; |q_inflation_pct| < 50%; action_std_spatial not collapsed.
NOT pass/fail: corr(u, rain_fwd7), yield (single-seed yield is noise; corr is the wrong target).

DECISION: x1 drops -> coverage was the bottleneck (Path 3 is the fix). x1 flat but frac_low_action_wet>0 -> coverage achieved, policy still stuck -> go to Path 1 (TD3). Training destabilises -> argues for TD3 target smoothing (Path 1).


In [ ]:
# Clone repo and install deps (SB3 pinned to 2.6.0 — same training contract as v2.14-v2.16).
import subprocess, sys, os

WORK = '/kaggle/working'
repo = os.path.join(WORK, 'thesis')
if os.path.exists(repo):
    subprocess.run(['rm', '-rf', repo], check=True)
subprocess.run(
    ['git', 'clone', 'https://github.com/taratorbati/thesis.git', repo],
    check=True)

os.chdir(repo)
sys.path.insert(0, repo)

subprocess.run(
    ['pip', 'install', '--quiet',
     'stable-baselines3==2.6.0', 'gymnasium', 'wandb', 'pytest'],
    check=True)

import torch
print(f'PyTorch:        {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:            {torch.cuda.get_device_name(0)}')


In [ ]:
# WandB secret + GPU check.
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')
    print('OK  WANDB_API_KEY loaded from Kaggle Secrets.')
except Exception as e:
    print(f'NOTE: Could not load WANDB_API_KEY ({type(e).__name__}). Training continues without WandB.')

import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'nvidia-smi failed - no GPU allocated')


In [ ]:
# Pre-training validation: smoke tests + 1000-step pilot of v2.17-P3 wiring.
# This pilot specifically exercises the NEW code paths: the NormalActionNoise
# constructor, the ExplorationNoiseDecayCallback, and the LowActionCoverageCallback.
import subprocess, sys

print('Smoke tests...')
r = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_rl_smoke.py', '-v', '--tb=short'],
    capture_output=False)
assert r.returncode == 0, 'SMOKE TESTS FAILED'

print('\nFactorized-critic tests (v2.7 through v2.16)...')
r = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_factorized_critic.py', '-v', '--tb=short'],
    capture_output=False)
assert r.returncode == 0, 'FACTORIZED CRITIC TESTS FAILED'

print('\n1000-step pilot training (wiring check for v2.17-P3, ~1-2 min)...')
from src.rl.train_v217_p3 import train_sac_v217_p3
_ = train_sac_v217_p3(
    seed=999,
    output_dir='/kaggle/working/pilot',
    wandb_project=None,
    total_timesteps=1000,
    explore_decay_steps=500,   # fast decay so the pilot exercises both phases
)
print('\nOK  Pre-flight passed (action-noise + decay + coverage callbacks wired). Proceed.')


In [ ]:
# Full 250k training (SAC v2.17-P3 — alpha=0.005 + decaying exploration noise).
# ~30-55 min on A100, ~2-2.5 h on T4.
#
# Defaults (set in train_v217_p3.py):
#   ent_coef=0.005, explore_sigma_start=0.30, explore_sigma_end=0.0,
#   explore_decay_steps=60000.  Override here if you want to retune without
#   editing the file (e.g. drop sigma_start to 0.20 if early critic_loss spikes).
#
# Start with SEED=0 (paired with v2.7/v2.11/v2.14/v2.15/v2.16 seed 0).

SEED = 0       # CHANGE per session

from src.rl.train_v217_p3 import train_sac_v217_p3

model = train_sac_v217_p3(
    seed=SEED,
    output_dir='/kaggle/working/thesis/results/rl',
    wandb_project='sac-irrigation-thesis',
    total_timesteps=250_000,
    gamma=0.99,
    actor_lr_mult=5.0,
    ent_coef=0.005,                 # *** v2.17-P3: lowered from 0.01 ***
    reward_overshoot_mode='linear', # carried from v2.15/v2.16
    rain_normaliser=30.0,           # carried from v2.16
    explore_sigma_start=0.30,       # *** v2.17-P3 exploration noise ***
    explore_sigma_end=0.0,
    explore_decay_steps=60_000,
)
print('Training complete.')


In [ ]:
# Archive results so Kaggle persists them after the session.
import shutil, os, datetime
src = f'/kaggle/working/thesis/results/rl/sac_v217_p3_seed{SEED}'
ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
dst = f'/kaggle/working/sac_v217_p3_seed{SEED}_{ts}'
shutil.copytree(src, dst, ignore=shutil.ignore_patterns('replay_buffer_latest.pkl'))
print(f'Archived to: {dst}  (download from the Kaggle output panel)')
for root, _, files in os.walk(dst):
    for f in files:
        p = os.path.join(root, f)
        print(f'  {os.path.relpath(p, dst)}  ({os.path.getsize(p)/1024:.1f} KB)')


In [ ]:
# Post-training 9-cell evaluation (SAC eval path, marker auto-dispatch).
# v2.17-P3 checkpoints carry marker=2.16, so runner.py dispatches to
# V216CTDESACPolicy and applies rain_normaliser=30.0 at eval automatically —
# NO eval-side change was needed for Path 3.
import subprocess, sys, os

model_path = f'/kaggle/working/thesis/results/rl/sac_v217_p3_seed{SEED}/best_model/best_model.zip'
final_path = f'/kaggle/working/thesis/results/rl/sac_v217_p3_seed{SEED}/sac_v217_p3_seed{SEED}_final.zip'

print('Evaluating BEST checkpoint on 9-cell grid (perfect forecast)...')
r = subprocess.run([
    sys.executable, '-m', 'scripts.experiments.exp_rl',
    '--mode', 'eval', '--model', model_path,
    '--scenario', 'all', '--budget', 'all', '--forecast', 'perfect',
    '--output-dir', f'/kaggle/working/thesis/results/runs/sac_v217_p3_best_model',
], capture_output=False)
assert r.returncode == 0, 'PERFECT-FORECAST EVAL (best) FAILED'

# Always eval the FINAL 250k checkpoint too — EvalCallback's "best" has
# historically been the 25k early checkpoint on a dry-skewed dev set.
if os.path.exists(final_path):
    print('\nEvaluating FINAL (250k) checkpoint for comparison...')
    subprocess.run([
        sys.executable, '-m', 'scripts.experiments.exp_rl',
        '--mode', 'eval', '--model', final_path,
        '--scenario', 'all', '--budget', 'all', '--forecast', 'perfect',
        '--output-dir', f'/kaggle/working/thesis/results/runs/sac_v217_p3_final_model',
    ], capture_output=False)


In [ ]:
# PRIMARY DIAGNOSTIC for Path 3 — "did wet-year over-irrigation drop?"
# Decide on x1 median + waterlog + water (NOT corr, NOT yield).
import pandas as pd, numpy as np, json, glob, os

BEST = '/kaggle/working/thesis/results/runs/sac_v217_p3_best_model'
roots = [BEST] + glob.glob('/kaggle/working/thesis/results/runs/sac_v217_p3*')
OUTPUT_DIR = next((d for d in roots if os.path.isdir(d)), BEST)
print('Reading eval outputs from:', OUTPUT_DIR)

def wet_aggregate(d):
    ys, ws, wls, x1s = [], [], [], []
    for bud in ['100pct', '85pct', '70pct']:
        pj = glob.glob(os.path.join(d, f'*wet*{bud}*seed0.json'))
        pq = glob.glob(os.path.join(d, f'*wet*{bud}*seed0.parquet'))
        if not pj or not pq:
            continue
        m = json.load(open(pj[0]))['final_metrics']
        df = pd.read_parquet(pq[0])
        ys.append(m['yield_kg_ha']); ws.append(m.get('water_used_mm'))
        wls.append(m.get('waterlog_days_per_agent')); x1s.append(float(df['x1'].median()))
    f = lambda a: float(np.mean([v for v in a if v is not None])) if a else float('nan')
    return f(ys), f(ws), f(wls), f(x1s)

y, w, wl, x1 = wet_aggregate(OUTPUT_DIR)
print('=' * 78)
print(' v2.17-P3 WET-YEAR aggregate (mean over 3 budgets)  vs references')
print('=' * 78)
print(f'{"metric":<22s} {"v2.17-P3":>9s} {"v2.16fix":>9s} {"autoA":>7s} {"MPC":>7s}  {"target":>10s}')
print(f'{"x1 median (mm)":<22s} {x1:9.1f} {152:9.0f} {132:7.0f} {130:7.0f}  {"< 140":>10s}')
print(f'{"waterlog days":<22s} {wl:9.1f} {76:9.0f} {35:7.0f} {20:7.0f}  {"< 55":>10s}')
print(f'{"water used (mm)":<22s} {w:9.0f} {397:9.0f} {324:7.0f} {309:7.0f}  {"< 360":>10s}')
print(f'{"yield (kg/ha)":<22s} {y:9.0f} {3391:9.0f} {3633:7.0f} {3752:7.0f}  {"(info)":>10s}')
print()
print('PRIMARY acceptance (decide here):')
print(f'  x1 median  < 140 : {"PASS" if x1 < 140 else "FAIL"}  ({x1:.1f})')
print(f'  waterlog   < 55  : {"PASS" if wl < 55  else "FAIL"}  ({wl:.1f})')
print(f'  water      < 360 : {"PASS" if w  < 360 else "FAIL"}  ({w:.0f})')

# Per-climate u_mean: is the policy finally responsive (wet u_mean << dry u_mean)?
def umean(d, scen, bud):
    pq = glob.glob(os.path.join(d, f'*{scen}*{bud}*seed0.parquet'))
    return float(pd.read_parquet(pq[0])['u'].mean()) if pq else float('nan')
du, wu = umean(OUTPUT_DIR, 'dry', '100pct'), umean(OUTPUT_DIR, 'wet', '100pct')
print(f'\n  dry/100 u_mean = {du:.2f} ; wet/100 u_mean = {wu:.2f} ; '
      f'gap = {du - wu:.2f} mm  (v2.16-fixed gap was only 0.37 mm)')


In [ ]:
# COVERAGE DIAGNOSTIC — did the injected noise actually reach low water?
# This is what makes a NULL result interpretable. If frac_low_action_wet stayed
# ~0 during exploration, the experiment failed to populate the buffer (retune
# sigma_start up) rather than proving the coverage hypothesis wrong.
import pandas as pd, glob, os
import matplotlib.pyplot as plt

run_dir = f'/kaggle/working/thesis/results/rl/sac_v217_p3_seed{SEED}'
cov = os.path.join(run_dir, 'low_action_coverage_log.csv')
sig = os.path.join(run_dir, 'exploration_sigma_log.csv')

if os.path.exists(cov):
    c = pd.read_csv(cov)
    print('Low-action coverage during training:')
    print(f"  peak frac_low_action      = {c['frac_low_action'].max():.3f}")
    if 'frac_low_action_wet' in c:
        wetc = c['frac_low_action_wet'].dropna()
        if len(wetc):
            print(f"  peak frac_low_action_wet  = {wetc.max():.3f}  "
                  f"(during {len(wetc)} logged wet-episode steps)")
            print(f"  -> coverage {'ACHIEVED' if wetc.max() > 0.0 else 'NOT achieved'}")
    print(f"  min action ever collected = {c['min_action'].min():.4f}  (0 = full low-water reach)")
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].plot(c['step'], c['frac_low_action'], lw=1, label='all states')
    if 'frac_low_action_wet' in c:
        ax[0].plot(c['step'], c['frac_low_action_wet'], '.', ms=3, label='wet states')
    ax[0].set_title('fraction of actions < 1 mm/day'); ax[0].set_xlabel('step')
    ax[0].legend(); ax[0].grid(alpha=0.3)
    if os.path.exists(sig):
        s = pd.read_csv(sig)
        ax[1].plot(s['step'], s['sigma'], lw=1)
        ax[1].set_title('exploration sigma schedule'); ax[1].set_xlabel('step')
        ax[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print('No coverage log found at', cov)


In [ ]:
# STABILITY DIAGNOSTIC — did alpha=0.005 + noise stay clean (no v2.7 cascade)?
import os, glob
import matplotlib.pyplot as plt
try:
    from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
    tb_dir = f'/kaggle/working/thesis/results/rl/sac_v217_p3_seed{SEED}/tensorboard'
    runs = glob.glob(os.path.join(tb_dir, '*'))
    assert runs, f'No tensorboard runs in {tb_dir}'
    ea = EventAccumulator(runs[0]); ea.Reload()
    tags = ea.Tags()['scalars']
    print('Available scalars:', sorted(tags))
    interesting = ['train/critic_loss', 'train/actor_loss',
                   'rollout/ep_rew_mean', 'v210/q_inflation_pct',
                   'v210/action_std_spatial', 'p3/exploration_sigma']
    available = [t for t in interesting if t in tags]
    if available:
        fig, axes = plt.subplots(2, 3, figsize=(15, 7))
        for i, tag in enumerate(available[:6]):
            ax = axes[i // 3, i % 3]
            ev = ea.Scalars(tag); xs = [e.step for e in ev]; ys = [e.value for e in ev]
            ax.plot(xs, ys, lw=1); ax.set_title(tag); ax.grid(alpha=0.3)
            if 'loss' in tag and ys and max(ys) > 100:
                ax.set_yscale('symlog')
        plt.tight_layout(); plt.show()
        cl = ea.Scalars('train/critic_loss') if 'train/critic_loss' in tags else []
        if cl:
            print(f"\n  max critic_loss = {max(e.value for e in cl):.2f}  "
                  f"(STABLE if < 100)")
except Exception as e:
    print('Tensorboard trajectory unavailable:', e)


In [ ]:
# Resume from a saved checkpoint (if the session was interrupted).
# Upload the prior Kaggle output as a dataset, then fill in the path.
#
# SEED = 0
# CHECKPOINT_STEP = 100_000
# CKPT = f'/kaggle/input/<your-dataset>/sac_v217_p3_seed{SEED}/checkpoints/sac_v217_p3_seed{SEED}_{CHECKPOINT_STEP}_steps.zip'
#
# from src.rl.train_v212 import AsymmetricLRSAC
# from src.rl.networks import V216CTDESACPolicy
# model = AsymmetricLRSAC.load(CKPT, custom_objects={'policy_class': V216CTDESACPolicy})
# # Continue: model.learn(total_timesteps=..., reset_num_timesteps=False)
# # NOTE: action_noise is NOT restored from a checkpoint; if resuming inside the
# # 60k exploration window, re-create NormalActionNoise + ExplorationNoiseDecayCallback.
